In [9]:
import os
import sys

root = os.path.abspath("..")
if root not in sys.path:
    sys.path.insert(0, root)

print("root:", root)
print(sys.path[:3])

root: /workspaces/churn_prediction-
['/workspaces/churn_prediction-', '/usr/local/python/3.12.1/lib/python312.zip', '/usr/local/python/3.12.1/lib/python3.12']


In [10]:
from app.scraper import load_data
from app.features import create_features

df = load_data()

df = create_features(df)

print(df.shape)

df.head()

(1040, 14)


,user_id,gender,age,total_products,total_quantity,total_amount,discounted_total,discount_ratio,avg_quantity_per_product,days_inactive,days_since_registration,large_cart,high_spender,churned
0,1,0,29,5,10,11799.309445,10992.891046,0.068345,2.000000,103,1432,1,1,0
1,1,0,29,3,11,13654.496663,11917.600845,0.127203,3.666667,349,107,0,1,1
2,1,0,29,5,10,13273.846104,10432.897877,0.214026,2.000000,271,1463,1,1,1
3,1,0,29,3,11,12340.772962,11745.673641,0.048222,3.666667,107,603,0,1,0
4,1,0,29,5,11,13601.167558,11974.292704,0.119613,2.200000,72,161,1,1,0


Method 1: Filter

In [12]:
from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    f_classif
)

import numpy as np

X = df.drop(
    columns=[
        "user_id",
        "churned"
    ]
)

y = df["churned"]

corr = X.corr().abs()

upper = corr.where(
    np.triu(
        np.ones(corr.shape),
        k=1
    ).astype(bool)
)

drop_cols = [
    c
    for c in upper.columns
    if any(
        upper[c] > 0.90
    )
]

X = X.drop(
    columns=drop_cols
)

selector = VarianceThreshold(
    threshold=0.01
)

X_var = pd.DataFrame(
    selector.fit_transform(X),
    columns=X.columns[
        selector.get_support()
    ]
)

anova = SelectKBest(
    f_classif,
    k=5
)

anova.fit(X_var, y)

filter_features = (
    X_var.columns[
        anova.get_support()
    ]
).tolist()

print(filter_features)

['age', 'days_inactive', 'days_since_registration', 'large_cart', 'high_spender']


Method 2: RFE

In [19]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

rfe = RFE(
    LogisticRegression(
        max_iter=1000,
    ),
    n_features_to_select=5
)

rfe.fit(X_scaled, y)

rfe_features = (
    X.columns[
        rfe.support_
    ]
).tolist()

print(rfe_features)

['total_products', 'total_quantity', 'total_amount', 'days_inactive', 'high_spender']


Method 3: Decision tree

In [20]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    random_state=42
)

dt.fit(X, y)

dt_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance":
        dt.feature_importances_
})

dt_features = (
    dt_importance
    .sort_values(
        "Importance",
        ascending=False
    )
    .head(5)
    ["Feature"]
    .tolist()
)

print(dt_features)

['days_inactive', 'age', 'gender', 'total_products', 'total_quantity']


Method 4: Random forest

In [21]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X, y)

rf_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance":
        rf.feature_importances_
})

rf_features = (
    rf_importance
    .sort_values(
        "Importance",
        ascending=False
    )
    .head(5)
    ["Feature"]
    .tolist()
)

print(rf_features)

['days_inactive', 'days_since_registration', 'total_amount', 'discount_ratio', 'total_quantity']


comparison table

In [22]:
comparison = pd.DataFrame({
    "Filter": pd.Series(
        filter_features
    ),
    "RFE": pd.Series(
        rfe_features
    ),
    "Decision Tree": pd.Series(
        dt_features
    ),
    "Random Forest": pd.Series(
        rf_features
    )
})

comparison

,Filter,RFE,Decision Tree,Random Forest
0,age,total_products,days_inactive,days_inactive
1,days_inactive,total_quantity,age,days_since_registration
2,days_since_registration,total_amount,gender,total_amount
3,large_cart,days_inactive,total_products,discount_ratio
4,high_spender,high_spender,total_quantity,total_quantity
